# Lesson 7: AI in SaaS

**Module 1:** Introduction to Artificial Intelligence

**Lesson Objectives:**
- Describe AI applications in SaaS
- Build a churn prediction model
- Evaluate business impact

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, precision_recall_curve)

print("Libraries loaded successfully")

## Generate Synthetic Customer Data

We simulate a SaaS company's customer database.

In [ ]:
np.random.seed(42)
n_customers = 5000

customers = pd.DataFrame({
    'customer_id': range(n_customers),
    'tenure_months': np.random.randint(1, 72, n_customers),
    'login_frequency_per_week': np.random.uniform(0, 20, n_customers),
    'support_tickets_last_6mo': np.random.randint(0, 20, n_customers),
    'features_used': np.random.randint(1, 25, n_customers),
    'days_since_last_login': np.random.randint(0, 120, n_customers),
    'monthly_spend': np.random.uniform(9, 199, n_customers),
    'plan': np.random.choice(['basic', 'pro', 'enterprise'], n_customers, 
                             p=[0.6, 0.3, 0.1]),
    'has_phone_support': np.random.choice([0, 1], n_customers, p=[0.7, 0.3]),
    'nps_score': np.random.randint(0, 11, n_customers),  # 0-10
    'email_open_rate': np.random.uniform(0, 1, n_customers),
})

# Create churn label based on realistic patterns
churn_prob = (
    (customers['login_frequency_per_week'] < 1).astype(float) * 0.3 +
    (customers['support_tickets_last_6mo'] > 10).astype(float) * 0.2 +
    (customers['days_since_last_login'] > 60).astype(float) * 0.3 +
    (customers['nps_score'] < 5).astype(float) * 0.15 +
    (customers['email_open_rate'] < 0.1).astype(float) * 0.1 +
    np.random.uniform(0, 0.1, n_customers)
)
customers['churn'] = (churn_prob > 0.3).astype(int)

print(f"Total customers: {len(customers)}")
print(f"Churned: {customers['churn'].sum()} ({customers['churn'].mean():.1%})")
print(f"\nFirst 5 rows:")
print(customers.head())

In [ ]:
# Exploratory analysis
print("Churn by Plan Type:")
print(customers.groupby('plan')['churn'].agg(['count', 'mean']))

print("\nChurn by Tenure (years):")
customers['tenure_years'] = customers['tenure_months'] // 12
print(customers.groupby('tenure_years')['churn'].agg(['count', 'mean']))

## Feature Engineering

In [ ]:
# Create engineered features
customers['login_per_month'] = customers['login_frequency_per_week'] * 4.33
customers['support_per_feature'] = customers['support_tickets_last_6mo'] / (customers['features_used'] + 1)
customers['spend_per_feature'] = customers['monthly_spend'] / (customers['features_used'] + 1)

# Encode categorical variables
le = LabelEncoder()
customers['plan_encoded'] = le.fit_transform(customers['plan'])

# Select features for model
feature_cols = [
    'tenure_months', 'login_frequency_per_week', 'support_tickets_last_6mo',
    'features_used', 'days_since_last_login', 'monthly_spend',
    'has_phone_support', 'nps_score', 'email_open_rate',
    'login_per_month', 'support_per_feature', 'spend_per_feature',
    'plan_encoded'
]

X = customers[feature_cols]
y = customers['churn']

print(f"Feature matrix: {X.shape}")
print(f"Features: {feature_cols}")

## Train and Compare Models

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Train churn rate: {y_train.mean():.1%}")
print(f"Test churn rate: {y_test.mean():.1%}")

In [ ]:
# Scale for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

# Train and evaluate each model
results = []
plt.figure(figsize=(10, 6))

for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    y_pred = (y_prob >= 0.5).astype(int)
    
    # Metrics
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        'Model': name,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'ROC-AUC': auc
    })
    
    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Churn Prediction')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

results_df = pd.DataFrame(results).set_index('Model')
print("Model Comparison:")
print(results_df.round(3))

## Feature Importance Analysis

Understanding which features drive churn helps design targeted interventions.

In [ ]:
rf_model = models['Random Forest']
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Churn Drivers:")
print(importance_df.head(10))

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'].head(10)[::-1], 
         importance_df['importance'].head(10)[::-1])
plt.xlabel('Feature Importance')
plt.title('Top 10 Factors Driving Customer Churn')
plt.tight_layout()
plt.show()

## Churn Risk Segmentation and Business Impact

Let us segment customers by churn risk and calculate potential savings.

In [ ]:
# Predict probabilities for all customers
all_prob = rf_model.predict_proba(X)[:, 1]
customers['churn_probability'] = all_prob

# Create risk segments
def risk_segment(prob):
    if prob < 0.2:
        return 'Low'
    elif prob <= 0.5:
        return 'Medium'
    else:
        return 'High'

customers['risk_segment'] = customers['churn_probability'].apply(risk_segment)

print("Churn Risk Distribution:")
segment_counts = customers['risk_segment'].value_counts()
for seg, count in segment_counts.items():
    actual_churn = customers[customers['risk_segment'] == seg]['churn'].mean()
    print(f"  {seg:8s}: {count:4d} customers (actual churn: {actual_churn:.1%})")

# Business impact calculation
high_risk = customers[customers['risk_segment'] == 'High']
monthly_revenue_at_risk = high_risk['monthly_spend'].sum()
print(f"\nMonthly revenue at risk (High risk): ${monthly_revenue_at_risk:,.0f}")

# Assume 50% of high-risk churners can be saved with intervention
save_rate = 0.5
annual_savings = monthly_revenue_at_risk * 12 * save_rate * 0.8  # 80% of high risk would actually churn
print(f"Estimated annual savings from churn interventions: ${annual_savings:,.0f}")

## Intervention Strategies by Segment

| Segment | Risk | Action |
|---|---|---|
| **Low** | <20% | Nurture: regular engagement, upsell opportunities |
| **Medium** | 20-50% | Engage: personalized email, feature tutorials, check-in call |
| **High** | >50% | Save: discount offer, account review, executive outreach |

In [ ]:
# Visualize churn probability distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist([
    customers[customers['churn'] == 0]['churn_probability'],
    customers[customers['churn'] == 1]['churn_probability']
], bins=30, label=['Retained', 'Churned'], alpha=0.7, edgecolor='black')
plt.xlabel('Predicted Churn Probability')
plt.ylabel('Count')
plt.title('Churn Probability Distribution')
plt.legend()

plt.subplot(1, 2, 2)
segment_order = ['Low', 'Medium', 'High']
segment_churn = [customers[customers['risk_segment'] == s]['churn'].mean() 
                 for s in segment_order]
plt.bar(segment_order, segment_churn, color=['green', 'orange', 'red'])
plt.xlabel('Risk Segment')
plt.ylabel('Actual Churn Rate')
plt.title('Actual Churn Rate by Risk Segment')
plt.axhline(y=customers['churn'].mean(), color='blue', linestyle='--', 
            label=f'Overall ({customers["churn"].mean():.1%})')
plt.legend()

plt.tight_layout()
plt.show()

## Exercises

1. Try a different classifier (e.g., SVM or XGBoost) and compare performance.
2. Optimize the probability threshold to maximize business value (consider cost of intervention vs. cost of churn).
3. Create a customer retention dashboard mockup showing key metrics and AI predictions.

In [ ]:
# Exercise 1: XGBoost comparison
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_estimators=100, random_state=42)
    xgb.fit(X_train, y_train)
    xgb_prob = xgb.predict_proba(X_test)[:, 1]
    xgb_auc = roc_auc_score(y_test, xgb_prob)
    print(f"XGBoost ROC-AUC: {xgb_auc:.3f}")
except ImportError:
    print("XGBoost not installed. Try: pip install xgboost")

## Summary

- AI powers churn prediction, personalization, marketing analytics, and product analytics in SaaS
- We built a churn prediction pipeline with feature engineering, model comparison, and business impact analysis
- Random Forest and Gradient Boosting perform well for churn prediction
- Feature importance reveals drivers of churn → enables targeted interventions
- Risk segmentation guides retention strategy (Low: nurture, Medium: engage, High: save)
- The business value of churn prediction is substantial: millions in recovered revenue